In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os
os.listdir('/content/drive/MyDrive/OULAD_processed')

In [ ]:
import pandas as pd
OUTPUT_PATH = '/content/drive/MyDrive/OULAD_processed/'


# ════════════════════════════════════════════════════════════
# MODULE M4 — EXPLAINABILITY (XAI)
# SHAP + LIME + Traduction pédagogique
# ════════════════════════════════════════════════════════════

In [ ]:
import pickle

with open(
    OUTPUT_PATH + "models_static/static_features_v2.pkl",
    "rb"
) as f:
    static_features_v2 = pickle.load(f)

print(static_features_v2)

In [ ]:
final_static_df = pd.read_csv(OUTPUT_PATH + 'oulad_final_static.csv')

# recréer les 3 features
final_static_df['is_retaker'] = (
    final_static_df['num_of_prev_attempts'] > 0
).astype(int)

final_static_df['credits_per_week'] = (
    final_static_df['studied_credits'] /
    (final_static_df['module_presentation_length'] / 7)
)

final_static_df['very_late_registration'] = (
    final_static_df['registration_lead_time'] < 14
).astype(int)

X_static_v2 = final_static_df[static_features_v2].fillna(0).astype(float).values
y_static_v2 = final_static_df['at_risk'].values

X_tr, X_te, y_tr, y_te = train_test_split(
    X_static_v2, y_static_v2,
    test_size=0.15,
    stratify=y_static_v2,
    random_state=SEED
)

X_tr, X_val, y_tr, y_val = train_test_split(
    X_tr, y_tr,
    test_size=0.15,
    stratify=y_tr,
    random_state=SEED
)

In [ ]:
import os, pickle, json, joblib
import numpy as np
import pandas as pd
import torch

# chemins
STATIC_PATH = OUTPUT_PATH + "models_static/"
DYN_PATH    = OUTPUT_PATH + "dynamic_profiling_kmeans_fixed_labels3/"
DYN_PATH1    = OUTPUT_PATH + "dynamic_profiling_kmeans_fixed_labels/"
DYN_PATH2    = OUTPUT_PATH + "dynamic_profiling_kmeans/"

# 1. Modèle statique XGBoost
xgb_final = joblib.load(STATIC_PATH + "xgb_optuna_static_day0.pkl")

with open(STATIC_PATH + "static_features_v2.pkl", "rb") as f:
    static_features_v2 = pickle.load(f)

with open(STATIC_PATH + "xgb_optuna_static_metrics.json", "r") as f:
    metrics_static = json.load(f)

# 2. Modèle GRU full
gru_weights = torch.load(
    OUTPUT_PATH + "best_gru_model1.pt",
    map_location=device
)

model.load_state_dict(gru_weights)
model.eval()

# 3. Modèles GRU checkpoints
checkpoint_weights = {}

for cp in CHECKPOINTS:
    checkpoint_weights[cp] = torch.load(
        OUTPUT_PATH + f"best_gru_checkpoint_w{cp}1.pt",
        map_location=device
    )

# 4. Risk scores full + test
risk_scores_full = np.load(OUTPUT_PATH + "gru_risk_scores_full.npy")

gru_full_test_results = pd.read_csv(
    OUTPUT_PATH + "gru_full_test_risk_scores.csv"
)

# 5. Risk scores checkpoints
with open(OUTPUT_PATH + "gru_checkpoint_risk_scores.pkl", "rb") as f:
    risk_scores_cp = pickle.load(f)

checkpoint_summary = pd.read_csv(
    OUTPUT_PATH + "gru_ews_checkpoint_summary.csv"
)

checkpoint_predictions = {}

for cp in CHECKPOINTS:
    checkpoint_predictions[cp] = pd.read_csv(
        OUTPUT_PATH + f"gru_checkpoint_w{cp}_predictions.csv"
    )

# 6. Embeddings
temporal_embeddings = np.load(
    OUTPUT_PATH + "temporal_embeddings.npy"
)

with open(OUTPUT_PATH + "gru_embeddings_checkpoints.pkl", "rb") as f:
    embeddings_cp = pickle.load(f)

with open(OUTPUT_PATH + "gru_embedding_metadata.pkl", "rb") as f:
    embedding_metadata = pickle.load(f)

student_ids_embeddings = pd.read_csv(
    OUTPUT_PATH + "student_ids_embeddings.csv"
)

# 7. Dynamic profiling
final_df_wcdmacp = pd.read_csv(
    DYN_PATH + "final_df_with_kmeans_profiles.csv"
)

pcea_df = pd.read_csv(
    DYN_PATH + "pcea_kmeans_results.csv"
)

with open(DYN_PATH1 + "kmeans_full.pkl", "rb") as f:
    km_full = pickle.load(f)

with open(DYN_PATH1 + "kmeans_checkpoints.pkl", "rb") as f:
    km_cp = pickle.load(f)

with open(DYN_PATH + "cluster_label_mappings.json", "r") as f:
    cluster_label_mappings = json.load(f)

with open(DYN_PATH2 + "clustering_metrics.pkl", "rb") as f:
    clustering_metrics = pickle.load(f)

emb_full_norm = np.load(DYN_PATH2 + "emb_full_norm.npy")
profiles_full = np.load(DYN_PATH2 + "profiles_full.npy")

with open(DYN_PATH2 + "emb_norm_cp.pkl", "rb") as f:
    emb_norm_cp = pickle.load(f)

with open(DYN_PATH2 + "profiles_cp.pkl", "rb") as f:
    profiles_cp = pickle.load(f)

risk_ladder_final = pd.read_csv(DYN_PATH + "risk_ladder_final.csv")
cluster_means_labeled = pd.read_csv(DYN_PATH + "cluster_means_labeled.csv")

print("Chargement M4/XAI terminé")
print("XGBoost features :", len(static_features_v2))
print("Risk scores full :", risk_scores_full.shape)
print("Temporal embeddings :", temporal_embeddings.shape)
print("Final profiling df :", final_df_wcdmacp.shape)

In [ ]:


embeddings_full = np.load(
    OUTPUT_PATH + "gru_embeddings_full.npy"
)

In [ ]:
!pip install shap lime --quiet

import shap
import lime
import lime.lime_tabular
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import warnings
warnings.filterwarnings('ignore')

print("=" * 60)
print("MODULE M4 — EXPLAINABILITY (XAI)")
print("SHAP + LIME + Traduction pédagogique")
print("=" * 60)


# ════════════════════════════════════════════════════════════
# CONFIGURATION
# ════════════════════════════════════════════════════════════

student_id_cols  = ['id_student', 'code_module', 'code_presentation']
CHECKPOINTS      = [3, 7, 12]
SEED             = 42

# Noms lisibles des features statiques
feature_names_readable = {
    'num_of_prev_attempts'     : 'Tentatives précédentes',
    'studied_credits'          : 'Crédits étudiés',
    'imd_score'                : 'Indice de précarité (IMD)',
    'gender_M'                 : 'Genre masculin',
    'disability_Y'             : 'Handicap déclaré',
    'highest_education_num'    : 'Niveau éducation antérieur',
    'age_band_num'             : 'Tranche d\'âge',
    'module_presentation_length': 'Durée du module (jours)',
    'registration_lead_time'   : 'Anticipation inscription (jours)',
    'registration_missing'     : 'Inscription tardive/atypique',
    'is_retaker'               : 'Étudiant récidiviste',
    'credits_per_week'         : 'Crédits par semaine',
    'very_late_registration'   : 'Inscription très tardive',
}
for col in [c for c in static_features_v2
            if c.startswith('module_')]:
    mod = col.replace('module_', '')
    feature_names_readable[col] = f'Module {mod}'

# Noms lisibles des features GRU (séquences)
seq_feature_names_readable = {
    'weekly_clicks_capped'         : 'Clics totaux (semaine)',
    'weekly_active_days'           : 'Jours actifs (semaine)',
    'weekly_unique_res'            : 'Ressources uniques visitées',
    'weekly_act_types'             : 'Types activités distincts',
    'weekly_mean_score'            : 'Score moyen (semaine)',
    'weekly_n_submitted'           : 'Devoirs soumis (semaine)',
    'weekly_n_late'                : 'Soumissions en retard',
    'weekly_content_clicks'        : 'Clics contenu pédagogique',
    'weekly_assessment_clicks'     : 'Clics activités évaluation',
    'weekly_social_clicks'         : 'Clics activités sociales',
    'weekly_resource_clicks'       : 'Clics ressources',
    'weekly_specialized_clicks'    : 'Clics activités spécialisées',
}


In [ ]:
# ════════════════════════════════════════════════════════════
# PARTIE A — SHAP SUR XGBOOST (MODÈLE STATIQUE JOUR 0)
# ════════════════════════════════════════════════════════════
print("\n" + "=" * 60)
print("PARTIE A — SHAP XGBoost (Jour 0)")
print("=" * 60)

# ── A1. Calcul SHAP ───────────────────────────────────────────
explainer_xgb   = shap.TreeExplainer(xgb_final)
shap_values_xgb = explainer_xgb.shap_values(X_te)

# Pour XGBoost → shap_values est un array direct
if isinstance(shap_values_xgb, list):
    shap_vals_xgb = shap_values_xgb[1]
else:
    shap_vals_xgb = shap_values_xgb

# Noms lisibles
readable_names = [feature_names_readable.get(f, f)
                  for f in static_features_v2]

print(f"Shape SHAP values : {shap_vals_xgb.shape}")
print(f"Expected value    : {explainer_xgb.expected_value:.4f}")

# ── A2. Summary Plot (beeswarm) ───────────────────────────────
plt.figure(figsize=(11, 7))
shap.summary_plot(
    shap_vals_xgb, X_te,
    feature_names = readable_names,
    show          = False,
    max_display   = 15
)
plt.title('SHAP Summary — XGBoost Jour 0\n'
          'Impact des features démographiques sur le risque',
          fontweight='bold', fontsize=13)
plt.tight_layout()
plt.show()

# ── A3. Bar Plot (importance globale) ────────────────────────
plt.figure(figsize=(10, 6))
shap.summary_plot(
    shap_vals_xgb, X_te,
    feature_names = readable_names,
    plot_type     = 'bar',
    show          = False,
    max_display   = 15
)
plt.title('SHAP Feature Importance — XGBoost Jour 0',
          fontweight='bold', fontsize=13)
plt.tight_layout()
plt.show()

# ── A4. Dependence Plots — Top 3 features ────────────────────
mean_abs_shap = np.abs(shap_vals_xgb).mean(axis=0)
top3_idx      = np.argsort(mean_abs_shap)[::-1][:3]
top3_names    = [static_features_v2[i] for i in top3_idx]
top3_readable = [readable_names[i] for i in top3_idx]

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle('SHAP Dependence Plots — Top 3 Features (XGBoost Jour 0)',
             fontweight='bold', fontsize=13)

for ax, feat_idx, feat_name in zip(axes, top3_idx, top3_readable):
    shap.dependence_plot(
        feat_idx, shap_vals_xgb, X_te,
        feature_names = readable_names,
        ax=ax, show=False
    )
    ax.set_title(feat_name, fontweight='bold')

plt.tight_layout()
plt.show()

# Waterfall
# Étudiant à risque
# Étudiant à risque
idx_risk = np.where(y_te == 1)[0][0]

shap_exp_risk = shap.Explanation(
    values=shap_vals_xgb[idx_risk],
    base_values=explainer_xgb.expected_value,
    data=X_te[idx_risk],
    feature_names=readable_names
)

shap.plots.waterfall(shap_exp_risk, max_display=10, show=False)
plt.title("Étudiant À Risque", fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()


# Étudiant non à risque
idx_safe = np.where(y_te == 0)[0][0]

shap_exp_safe = shap.Explanation(
    values=shap_vals_xgb[idx_safe],
    base_values=explainer_xgb.expected_value,
    data=X_te[idx_safe],
    feature_names=readable_names
)

shap.plots.waterfall(shap_exp_safe, max_display=10, show=False)
plt.title("Étudiant Non À Risque", fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
import os

os.makedirs(
    OUTPUT_PATH + "xai/",
    exist_ok=True
)

In [ ]:
# =====================================================
# SAUVEGARDE SHAP XGBOOST
# =====================================================

np.save(
    OUTPUT_PATH + "xai/shap_values_xgb.npy",
    shap_vals_xgb
)

np.save(
    OUTPUT_PATH + "xai/xgb_test_features.npy",
    X_te
)

np.save(
    OUTPUT_PATH + "xai/xgb_test_labels.npy",
    y_te
)

with open(
    OUTPUT_PATH + "xai/xgb_feature_names.pkl",
    "wb"
) as f:
    pickle.dump(readable_names, f)

print("SHAP XGBoost sauvegardé")

In [ ]:
idx_test = np.load(
    OUTPUT_PATH + 'idx_test.npy'
)

print(idx_test.shape)

In [ ]:
# ════════════════════════════════════════════════════════════
# PARTIE B — SHAP SUR GRU (MODÈLE TEMPOREL)
# Approximation via GradientExplainer
# ════════════════════════════════════════════════════════════
print("\n" + "=" * 60)
print("PARTIE B — SHAP GRU (Modèle temporel)")
print("=" * 60)

# Pour les modèles deep learning, on utilise GradientExplainer
# qui approxime les SHAP values via les gradients

# Préparer les données de background (100 exemples représentatifs)
np.random.seed(SEED)
background_idx = np.random.choice(len(X_train), 100, replace=False)
X_background   = torch.tensor(
    X_train[background_idx], dtype=torch.float32
).to(device)

# Données de test
X_test_tensor  = torch.tensor(
    X_test[:200], dtype=torch.float32
).to(device)

# Wrapper pour extraire uniquement le risk score
class GRURiskWrapper(torch.nn.Module):
    def __init__(self, gru_model):
        super().__init__()
        self.model = gru_model

    def forward(self, x):
        _, risk = self.model(x)
        return risk


model.train()
model_wrapper = GRURiskWrapper(model).to(device)
model_wrapper.train()

# SHAP GradientExplainer
print("Calcul SHAP GradientExplainer (GRU)...")

# Désactiver cuDNN seulement pendant SHAP
with torch.backends.cudnn.flags(enabled=False):

    explainer_gru = shap.GradientExplainer(
        model_wrapper,
        X_background
    )

    shap_values_gru = explainer_gru.shap_values(
        X_test_tensor
    )

# Shape : (N, 38, F) → moyenner sur les semaines

print(f"Shape SHAP GRU : {np.array(shap_values_gru).shape}")

shap_values_gru = np.array(shap_values_gru)

# Supprimer la dernière dimension si elle existe
if shap_values_gru.ndim == 4:
    shap_values_gru = shap_values_gru.squeeze(-1)

print("Shape SHAP GRU corrigée :", shap_values_gru.shape)

# Moyenner sur les semaines → importance par feature
shap_gru_mean = np.abs(
    shap_values_gru
).mean(axis=1)  # (N, F)

shap_gru_global = shap_gru_mean.mean(axis=0)  # (F,)

# Noms des features séquentielles
seq_feature_names = [
    'weekly_clicks_capped', 'weekly_active_days',
    'weekly_unique_res', 'weekly_act_types',
    'weekly_mean_score', 'weekly_n_submitted',
    'weekly_n_late', 'weekly_content_clicks',
    'weekly_assessment_clicks', 'weekly_social_clicks',
    'weekly_resource_clicks', 'weekly_specialized_clicks',
][:F]

readable_seq = [seq_feature_names_readable.get(f, f)
                for f in seq_feature_names]

# ── B1. Importance globale des features GRU ───────────────────
fig, ax = plt.subplots(figsize=(10, 6))
sorted_idx = np.argsort(shap_gru_global)
ax.barh(
    [readable_seq[i] for i in sorted_idx],
    shap_gru_global[sorted_idx],
    color='#7C3AED', alpha=0.85
)
ax.set_xlabel('|SHAP| moyen')
ax.set_title('SHAP Feature Importance — GRU\n'
             '(moyenne sur toutes les semaines)',
             fontweight='bold', fontsize=13)
ax.grid(alpha=0.3, axis='x')
plt.tight_layout()
plt.show()

# ── B2. Importance SHAP par semaine (heatmap temporelle) ──────
shap_gru_temporal = np.abs(
    shap_values_gru
).mean(axis=0)  # (38, F) → moyenne sur étudiants

fig, ax = plt.subplots(figsize=(16, 6))
shap_temporal_df = pd.DataFrame(
    shap_gru_temporal.T,
    index   = readable_seq,
    columns = [f'Sem.{w}' for w in range(38)]
)
sns.heatmap(
    shap_temporal_df,
    cmap='YlOrRd', ax=ax,
    cbar_kws={'label': '|SHAP| moyen'}
)
ax.set_title(
    'SHAP temporel — Impact des features par semaine (GRU)',
    fontweight='bold', fontsize=13
)
ax.set_xlabel('Semaine')
ax.set_ylabel('Feature')
for cp in CHECKPOINTS:
    ax.axvline(x=cp, color='blue', ls='--',
               alpha=0.6, lw=1.5)
plt.tight_layout()
plt.show()

# ── B3. SHAP par checkpoint ───────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(18, 6))
fig.suptitle('SHAP GRU — Importance par checkpoint',
             fontweight='bold', fontsize=13)

for ax, cp in zip(axes, CHECKPOINTS):
    shap_cp = np.abs(
        shap_values_gru[:, :cp, :]
    ).mean(axis=(0, 1))  # moyenne sur étudiants et semaines
    sorted_idx = np.argsort(shap_cp)
    ax.barh(
        [readable_seq[i] for i in sorted_idx],
        shap_cp[sorted_idx],
        color='#2563EB', alpha=0.85
    )
    ax.set_title(f'Semaine {cp}', fontweight='bold')
    ax.set_xlabel('|SHAP| moyen')
    ax.grid(alpha=0.3, axis='x')

plt.tight_layout()
plt.show()

# Remettre le modèle en mode évaluation après SHAP
model_wrapper.eval()
model.eval()

In [ ]:
# =====================================================
# SAUVEGARDE SHAP GRU GLOBAL
# =====================================================

np.save(
    OUTPUT_PATH + "xai/shap_gru_full.npy",
    shap_values_gru
)

np.save(
    OUTPUT_PATH + "xai/shap_gru_global.npy",
    shap_gru_global
)

print("SHAP GRU global sauvegardé")

In [ ]:
# ════════════════════════════════════════════════════════════
# PARTIE C — LIME (Explications locales)
# ════════════════════════════════════════════════════════════
print("\n" + "=" * 60)
print("PARTIE C — LIME (Explications locales)")
print("=" * 60)

# ── C1. LIME sur XGBoost (Jour 0) ────────────────────────────
print("\nLIME — XGBoost Jour 0")

lime_explainer_xgb = lime.lime_tabular.LimeTabularExplainer(
    training_data   = X_tr,
    feature_names   = readable_names,
    class_names     = ['Non à risque', 'À risque'],
    mode            = 'classification',
    discretize_continuous = True,
    random_state    = SEED
)

# Expliquer 2 étudiants : 1 à risque + 1 non à risque
lime_cases = [
    (np.where(y_te == 1)[0][0], 'À Risque'),
    (np.where(y_te == 0)[0][0], 'Non À Risque'),
]

for idx, case_label in lime_cases:
    print(f"\nLIME — Étudiant {case_label} :")

    exp = lime_explainer_xgb.explain_instance(
        data_row        = X_te[idx],
        predict_fn      = xgb_final.predict_proba,
        num_features    = 10,
        num_samples     = 1000,
    )

    risk_score = xgb_final.predict_proba(
        X_te[idx:idx+1]
    )[0, 1]
    print(f"  Risk score prédit : {risk_score:.4f}")

    # Affichage LIME
    fig = exp.as_pyplot_figure(label=1)
    fig.set_size_inches(10, 6)
    plt.title(
        f'LIME — XGBoost Jour 0\nÉtudiant {case_label} '
        f'(risk={risk_score:.3f})',
        fontweight='bold'
    )
    plt.tight_layout()
    plt.show()

    # Explication texte
    print(f"  Top features LIME :")
    for feat, weight in exp.as_list(label=1)[:5]:
        direction = '↑ risque' if weight > 0 else '↓ risque'
        print(f"    {feat:<45} → {weight:+.4f} ({direction})")

In [ ]:
# ════════════════════════════════════════════════════════════
# PARTIE D — TRADUCTION PÉDAGOGIQUE XGBOOST
# ════════════════════════════════════════════════════════════
print("\n" + "=" * 60)
print("PARTIE D — Traduction pédagogique")
print("=" * 60)

OPTIMAL_THRESHOLD = 0.5

# Règles de traduction SHAP → message pédagogique
PEDAGOGICAL_RULES = {
    'Niveau éducation antérieur': {
        'direction': 'negative',
        'high': "Bon niveau d'éducation antérieur — facteur protecteur",
        'low' : "Niveau d'éducation antérieur faible — "
                "suivi renforcé recommandé"
    },
    'Crédits par semaine': {
        'direction': 'positive',
        'high': "Charge de travail très élevée — "
                "risque de surcharge cognitive",
        'low' : "Charge de travail adaptée"
    },
    'Indice de précarité (IMD)': {
        'direction': 'positive',
        'high': "Contexte socio-économique défavorable — "
                "envisager un accompagnement",
        'low' : "Contexte socio-économique favorable"
    },
    'Anticipation inscription (jours)': {
        'direction': 'negative',
        'high': "Inscription anticipée — signe de motivation",
        'low' : "Inscription tardive — engagement à surveiller"
    },
    'Tentatives précédentes': {
        'direction': 'positive',
        'high': "Étudiant récidiviste — a déjà échoué ce cours",
        'low' : "Première tentative"
    },
    'Clics totaux (semaine)': {
        'direction': 'negative',
        'high': "Engagement VLE élevé cette semaine",
        'low' : "Faible engagement VLE — absence d'activité détectée"
    },
    'Score moyen (semaine)': {
        'direction': 'negative',
        'high': "Bonnes performances aux évaluations",
        'low' : "Performances faibles — révision nécessaire"
    },
    'Devoirs soumis (semaine)': {
        'direction': 'negative',
        'high': "Devoirs soumis régulièrement",
        'low' : "Aucun devoir soumis cette semaine"
    },
    'Soumissions en retard': {
        'direction': 'positive',
        'high': "Plusieurs soumissions en retard — "
                "problème d'organisation",
        'low' : "Pas de retard détecté"
    },
}

pedagogical_reports = []

def generate_pedagogical_explanation(
    student_idx, shap_vals, feature_names,
    feature_values, risk_score, model_name="XGBoost Jour 0"
):
    """
    Génère une explication pédagogique pour un étudiant.

    - Seuil de risque = OPTIMAL_THRESHOLD
    - Seuil d'impact relatif (percentile 75) au lieu de fixe (0.1)
    - Recommandation alignée sur OPTIMAL_THRESHOLD
    """
    print(f"\n{'='*55}")
    print(f"RAPPORT PÉDAGOGIQUE — {model_name}")
    print(f"{'='*55}")

    # ──utiliser OPTIMAL_THRESHOLD et non 0.5
    risk_label = 'À risque' if risk_score >= OPTIMAL_THRESHOLD \
                 else 'Faible risque'
    print(f"Score de risque : {risk_score:.1%} ({risk_label})")
    print(f"\nFacteurs déterminants :")

    # Trier par impact absolu
    feat_impacts = sorted(
        zip(feature_names, shap_vals, feature_values),
        key=lambda x: abs(x[1]), reverse=True
    )[:6]

    # ──seuil d'impact relatif (percentile 75)
    abs_shap_vals  = [abs(s) for _, s, _ in feat_impacts]
    threshold_high = np.percentile(abs_shap_vals, 75)

    messages = []
    for feat, shap_val, feat_val in feat_impacts:
        rule = PEDAGOGICAL_RULES.get(feat)

        # ──direction intelligente selon le type de feature
        # Certaines features sont "protectrices"
        # ex: bon score, devoirs soumis, engagement élevé
        # Donc SHAP > 0 signifie souvent valeur FAIBLE = mauvais

        if rule:
          feat_direction = rule.get('direction', 'positive')

          if feat_direction == 'negative':
            # Feature protectrice :
            # SHAP > 0 = faible valeur = augmente le risque
            direction = 'low' if shap_val > 0 else 'high'
          else:
            # Feature normale :
            # SHAP > 0 = valeur élevée = augmente le risque
            direction = 'high' if shap_val > 0 else 'low'
        else:
          direction = 'high' if shap_val > 0 else 'low'

        # Icône cohérente avec l'effet sur le risque
        direction_icon = '📈 ↑ risque' if shap_val > 0 \
                         else '📉 ↓ risque'


        if rule:
            msg = rule[direction]
        else:
            arrow = '↑ augmente' if shap_val > 0 else '↓ réduit'
            msg   = f"{feat} ({arrow} le risque)"

        # ──seuil relatif pour l'impact
        impact = '🔴 Fort impact' if abs(shap_val) >= threshold_high \
                 else '🟡 Impact modéré'

        print(f"  {impact} | {direction_icon} | {msg}")
        print(f"           Valeur : {feat_val:.3f} | "
              f"SHAP : {shap_val:+.4f}")
        messages.append(msg)

    print(f"\nRecommandation :")

    # ──recommandation alignée sur OPTIMAL_THRESHOLD
    if risk_score >= 0.7:
        print("  🔴 INTERVENTION URGENTE — Contacter l'étudiant "
              "immédiatement")
    elif risk_score >= OPTIMAL_THRESHOLD:
        print("  🟡 SURVEILLANCE RENFORCÉE — Proposer un suivi "
              "personnalisé")
    else:
        print("  🟢 FAIBLE RISQUE — Maintenir le suivi standard")

    report = {
        "risk_score": float(risk_score),
        "messages": messages
    }

    return report


# ════════════════════════════════════════════════════════════
# Sélection des exemples les plus représentatifs
# ════════════════════════════════════════════════════════════

# Prédictions sur tout le test set
y_proba_all = xgb_final.predict_proba(X_te)[:, 1]
y_pred_all  = (y_proba_all >= OPTIMAL_THRESHOLD).astype(int)

# ──exemples bien séparés et correctement classifiés

# Cas 1 — Vrai positif le plus confiant (score le plus élevé)
tp_idx      = np.where((y_te == 1) & (y_pred_all == 1))[0]
idx_high    = tp_idx[np.argmax(y_proba_all[tp_idx])]

# Cas 2 — Vrai positif modéré (score le plus proche du centre [seuil, 0.7])
tp_mod_idx  = np.where(
    (y_te == 1) &
    (y_pred_all == 1) &
    (y_proba_all >= OPTIMAL_THRESHOLD) &
    (y_proba_all < 0.7)
)[0]
if len(tp_mod_idx) > 0:
    mid_target  = (OPTIMAL_THRESHOLD + 0.7) / 2
    idx_mod     = tp_mod_idx[
        np.argmin(np.abs(y_proba_all[tp_mod_idx] - mid_target))
    ]
else:
    # Fallback : deuxième vrai positif le plus confiant
    idx_mod = tp_idx[np.argsort(y_proba_all[tp_idx])[-2]] \
              if len(tp_idx) >= 2 else tp_idx[0]

# Cas 3 — Vrai négatif le plus confiant (score le plus faible)
tn_idx      = np.where((y_te == 0) & (y_pred_all == 0))[0]
idx_low     = tn_idx[np.argmin(y_proba_all[tn_idx])]

# ════════════════════════════════════════════════════════════
# Génération des rapports
# ════════════════════════════════════════════════════════════
print("\nGénération des rapports pédagogiques...")

for test_idx, label in [
    (idx_high, 'À Risque (haut)'),
    (idx_mod,  'À Risque (moyen)'),
    (idx_low,  'Non À Risque'),
]:
    risk_score = float(xgb_final.predict_proba(
        X_te[test_idx:test_idx+1]
    )[0, 1])

    report = generate_pedagogical_explanation(
        student_idx   = test_idx,
        shap_vals     = shap_vals_xgb[test_idx],
        feature_names = readable_names,
        feature_values= X_te[test_idx],
        risk_score    = risk_score,
        model_name    = f"XGBoost Jour 0 — Cas {label}"
    )

    pedagogical_reports.append({
        "model": "xgboost",
        "case": label,
        "report": report
    })

In [ ]:
# ════════════════════════════════════════════════════════════
# RECONSTRUIRE checkpoint_results POUR M4/XAI
# ════════════════════════════════════════════════════════════

import pickle
import torch
import pandas as pd

checkpoint_results = {}

for cp in CHECKPOINTS:

    # 1. Recréer le modèle checkpoint
    model_cp = GRU_Pipeline(
        input_size  = F,
        hidden_size = HIDDEN_SIZE,
        n_layers    = 2,
        dropout     = 0.3
    ).to(device)

    # 2. Charger les poids sauvegardés
    weights_cp = torch.load(
        OUTPUT_PATH + f'best_gru_checkpoint_w{cp}1.pt',
        map_location=device
    )

    model_cp.load_state_dict(weights_cp)
    model_cp.eval()

    # 3. Charger les prédictions sauvegardées
    preds_df = pd.read_csv(
        OUTPUT_PATH + f'gru_checkpoint_w{cp}_predictions.csv'
    )

    # 4. Charger les métriques sauvegardées
    with open(
        OUTPUT_PATH + f'gru_checkpoint_w{cp}_metrics.pkl',
        'rb'
    ) as f:
        metrics_cp = pickle.load(f)

    # 5. Charger l'historique sauvegardé
    with open(
        OUTPUT_PATH + f'gru_checkpoint_w{cp}_history.pkl',
        'rb'
    ) as f:
        history_cp = pickle.load(f)

    # 6. Reconstruire checkpoint_results
    checkpoint_results[cp] = {
        'model'        : model_cp,
        'true'         : preds_df['y_true'].values,
        'preds'        : preds_df['risk_score'].values,
        'labels'       : preds_df['pred_label'].values,
        'best_val_auc' : metrics_cp['best_val_auc'],
        'auc'          : metrics_cp['test_auc'],
        'precision'    : metrics_cp['precision'],
        'recall'       : metrics_cp['recall'],
        'f1'           : metrics_cp['f1'],
        'history'      : history_cp,
        'threshold'    : metrics_cp['decision_threshold'],
    }

print("checkpoint_results reconstruit")
print(checkpoint_results.keys())

In [ ]:
# ════════════════════════════════════════════════════════════
# LIME + SHAP + TRADUCTION PÉDAGOGIQUE — GRU TEMPOREL
# ════════════════════════════════════════════════════════════

import shap
import lime
import lime.lime_tabular
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import warnings
warnings.filterwarnings('ignore')

SEED        = 42
CHECKPOINTS = [3, 7, 12]
OPTIMAL_THRESHOLD = 0.35

# Noms lisibles des features séquentielles
seq_feature_names = [
    'weekly_clicks_capped', 'weekly_active_days',
    'weekly_unique_res', 'weekly_act_types',
    'weekly_mean_score', 'weekly_n_submitted',
    'weekly_n_late', 'weekly_content_clicks',
    'weekly_assessment_clicks', 'weekly_social_clicks',
    'weekly_resource_clicks', 'weekly_specialized_clicks',
][:F]

readable_seq_list = [seq_feature_names_readable.get(f, f)
                     for f in seq_feature_names]

# Règles pédagogiques GRU
PEDAGOGICAL_RULES_GRU = {
    'Clics totaux (semaine)': {
        'high': "Engagement VLE élevé — étudiant actif",
        'low' : "Faible engagement VLE — absence d'activité"
    },
    'Jours actifs (semaine)': {
        'high': "Régularité de connexion — bonne habitude",
        'low' : "Connexions irrégulières — décrochage potentiel"
    },
    'Score moyen (semaine)': {
        'high': "Bonnes performances aux évaluations",
        'low' : "Performances faibles — révision nécessaire"
    },
    'Devoirs soumis (semaine)': {
        'high': "Devoirs soumis régulièrement",
        'low' : "Aucun devoir soumis cette semaine"
    },
    'Soumissions en retard': {
        'high': "Retards fréquents — problème d'organisation",
        'low' : "Pas de retard détecté"
    },
    'Clics activités évaluation': {
        'high': "Forte préparation aux évaluations",
        'low' : "Peu d'activité d'évaluation"
    },
    'Clics activités sociales': {
        'high': "Bonne participation sociale (forums)",
        'low' : "Peu d'interaction sociale — isolement potentiel"
    },
    'Ressources uniques visitées': {
        'high': "Exploration large des ressources",
        'low' : "Navigation limitée dans le cours"
    },
    'Clics contenu pédagogique': {
        'high': "Fort engagement avec le contenu du cours",
        'low' : "Peu d'accès au contenu pédagogique"
    },
    'Types activités distincts': {
        'high': "Diversité des activités — engagement varié",
        'low' : "Activité monotone — peu d'exploration"
    },
    'Clics ressources': {
        'high': "Utilisation active des ressources",
        'low' : "Peu d'utilisation des ressources"
    },
    'Clics activités spécialisées': {
        'high': "Engagement dans les activités spécialisées",
        'low' : "Peu d'activités spécialisées"
    },
}


# ════════════════════════════════════════════════════════════
# PARTIE B — SHAP SUR GRU PAR CHECKPOINT
# ════════════════════════════════════════════════════════════
print("=" * 60)
print("PARTIE B — SHAP GRU par checkpoint")
print("=" * 60)

# Wrapper GRU pour extraire uniquement le risk score
class GRURiskWrapper(torch.nn.Module):
    def __init__(self, gru_model):
        super().__init__()
        self.model = gru_model

    def forward(self, x):
        _, risk = self.model(x)
        return risk

model.train()

model_wrapper = GRURiskWrapper(model).to(device)
model_wrapper.train()

shap_results = {}  # stocke les SHAP par checkpoint

for cp in CHECKPOINTS:
    print(f"\n── SHAP GRU — Semaine {cp} ──────────────────────────")

    # Modèle checkpoint
    model_cp = checkpoint_results[cp]['model']

    class CheckpointWrapper(torch.nn.Module):
        def __init__(self, m):
            super().__init__()
            self.m = m
        def forward(self, x):
            _, risk = self.m(x)
            return risk



    wrapper_cp = CheckpointWrapper(model_cp).to(device)

    # Important pour GRU + SHAP
    model_cp.train()
    wrapper_cp.train()

    # Background (100 exemples)
    np.random.seed(SEED)
    bg_idx = np.random.choice(len(X_train), 100, replace=False)

    X_bg = torch.tensor(
        X_train[bg_idx, :cp, :],
        dtype=torch.float32
    ).to(device)

    # Test (200 exemples)
    X_test_cp = torch.tensor(
        X_test[:200, :cp, :],
        dtype=torch.float32
    ).to(device)

    # GradientExplainer
    print(f"  Calcul SHAP GradientExplainer (sem.{cp})...")
    with torch.backends.cudnn.flags(enabled=False):
      explainer_cp = shap.GradientExplainer(
          wrapper_cp,
          X_bg
      )

      shap_vals_cp = explainer_cp.shap_values(
          X_test_cp
      )

    # Remettre en mode évaluation après SHAP
    model_cp.eval()
    wrapper_cp.eval()

    shap_arr = np.array(shap_vals_cp)

    # Supprimer dernière dimension si shape = (N, cp, F, 1)
    if shap_arr.ndim == 4:
      shap_arr = shap_arr.squeeze(-1)


    # Importance globale : moyenne sur étudiants et semaines
    shap_global_cp = np.abs(shap_arr).mean(axis=(0, 1))  # (F,)
    shap_results[cp] = {
        'shap_arr'   : shap_arr,
        'shap_global': shap_global_cp,
        'X_test'     : X_test_cp.cpu().numpy(),
    }

    # ── B1. Bar plot importance ────────────────────────────
    fig, ax = plt.subplots(figsize=(10, 6))
    sorted_idx = np.argsort(shap_global_cp)
    ax.barh(
        [readable_seq_list[i] for i in sorted_idx],
        shap_global_cp[sorted_idx],
        color='#7C3AED', alpha=0.85
    )
    ax.set_xlabel('|SHAP| moyen')
    ax.set_title(
        f'SHAP Feature Importance — GRU\n'
        f'(moyenné sur {cp} semaines × 200 étudiants)',
        fontweight='bold', fontsize=12
    )
    ax.grid(alpha=0.3, axis='x')
    plt.tight_layout()
    plt.show()

    # Top 3
    top3 = np.argsort(shap_global_cp)[::-1][:3]
    print(f"  Top 3 features :")
    for i in top3:
        print(f"    {readable_seq_list[i]:<35} "
              f"SHAP={shap_global_cp[i]:.4f}")

# =====================================================
# SAUVEGARDE SHAP CHECKPOINTS
# =====================================================

with open(
    OUTPUT_PATH + "xai/shap_results_checkpoints.pkl",
    "wb"
) as f:
    pickle.dump(shap_results, f)

print("SHAP checkpoints sauvegardés")

# ── B2. Heatmap temporelle par checkpoint ─────────────────
fig, axes = plt.subplots(1, 3, figsize=(20, 6))
fig.suptitle('SHAP temporel — Impact par semaine et feature (GRU)',
             fontweight='bold', fontsize=13)

for ax, cp in zip(axes, CHECKPOINTS):
    shap_arr = shap_results[cp]['shap_arr']
    # (N, cp, F) → moyenne sur étudiants → (cp, F)
    shap_time = np.abs(shap_arr).mean(axis=0).T  # (F, cp)

    import seaborn as sns
    sns.heatmap(
        pd.DataFrame(
            shap_time,
            index   = readable_seq_list,
            columns = [f'Sem.{w}' for w in range(cp)]
        ),
        cmap='YlOrRd', ax=ax,
        cbar_kws={'label': '|SHAP|'}
    )
    ax.set_title(f'{cp}eme semaine', fontweight='bold')
    ax.set_xlabel('Semaine')
    ax.set_ylabel('')
    ax.tick_params(axis='x', rotation=45, labelsize=7)

plt.tight_layout()
plt.show()

# ── B3. Évolution de l'importance SHAP par checkpoint ─────
fig, ax = plt.subplots(figsize=(12, 6))
colors_feat = plt.cm.tab10(np.linspace(0, 1, F))

for i, (feat_name, color) in enumerate(
    zip(readable_seq_list, colors_feat)
):
    shap_per_cp = [shap_results[cp]['shap_global'][i]
                   for cp in CHECKPOINTS]
    ax.plot(CHECKPOINTS, shap_per_cp, 'o-',
            color=color, lw=2, ms=6, label=feat_name)

ax.set_xlabel('Nombre de semaines observées', fontsize=12)
ax.set_ylabel('|SHAP| moyen', fontsize=12)
ax.set_title('Évolution de l\'importance SHAP par checkpoint — GRU',
             fontweight='bold', fontsize=13)
ax.set_xticks(CHECKPOINTS)
ax.legend(fontsize=8, bbox_to_anchor=(1.05, 1),
          loc='upper left')
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()


model_wrapper.eval()
model.eval()


In [ ]:
# ════════════════════════════════════════════════════════════
# PARTIE C — LIME SUR GRU
# ════════════════════════════════════════════════════════════
print("\n" + "=" * 60)
print("PARTIE C — LIME GRU (Explications locales)")
print("=" * 60)

# LIME sur données tabulaires aplaties par checkpoint
# Shape entrée LIME : (N, cp × F)

lime_results = {}

for cp in CHECKPOINTS:
    print(f"\n── LIME GRU — Semaine {cp} ─────────────────────────")

    model_cp = checkpoint_results[cp]['model']
    model_cp.eval()

    # Données aplaties : (N, cp × F)
    X_train_cp_flat = X_train[:, :cp, :].reshape(
        len(X_train), -1
    )
    X_test_cp_flat = X_test[:200, :cp, :].reshape(
        200, -1
    )

    y_test_cp = y_test[:200]

    # Feature names aplaties : "Sem1_Clics", "Sem1_JoursActifs", ...
    flat_names = [
        f"Sem{w+1}_{readable_seq_list[f]}"
        for w in range(cp)
        for f in range(F)
    ]

    # Fonction predict pour LIME
    def predict_fn_cp(X_flat):
        X_3d = X_flat.reshape(-1, cp, F)
        X_t  = torch.tensor(X_3d, dtype=torch.float32).to(device)
        with torch.no_grad():
            _, risk = model_cp(X_t, None)
        proba = risk.cpu().numpy().flatten()
        return np.column_stack([1 - proba, proba])

    # LIME Explainer
    lime_explainer_gru = lime.lime_tabular.LimeTabularExplainer(
        training_data         = X_train_cp_flat,
        feature_names         = flat_names,
        class_names           = ['Non à risque', 'À risque'],
        mode                  = 'classification',
        discretize_continuous = True,
        random_state          = SEED
    )

    # Identifier 1 à risque + 1 non à risque
    risk_preds = checkpoint_results[cp]['preds']
    true_labels = checkpoint_results[cp]['true']

    at_risk_idx  = np.where(
        (risk_preds[:200] >= OPTIMAL_THRESHOLD) &
        (true_labels[:200] == 1)
    )[0]
    not_risk_idx = np.where(
        (risk_preds[:200] < OPTIMAL_THRESHOLD) &
        (true_labels[:200] == 0)
    )[0]

    lime_cases = []
    if len(at_risk_idx) > 0:
        lime_cases.append((at_risk_idx[0], 'À Risque'))
    if len(not_risk_idx) > 0:
        lime_cases.append((not_risk_idx[0], 'Non À Risque'))

    for idx, case_label in lime_cases:
        risk_score = float(risk_preds[idx])
        print(f"\n  Étudiant {case_label} | "
              f"Risk score = {risk_score:.4f}")

        exp = lime_explainer_gru.explain_instance(
            data_row   = X_test_cp_flat[idx],
            predict_fn = predict_fn_cp,
            num_features = 10,
            num_samples  = 500,
        )

        lime_results[(cp, idx)] = {
                "case": case_label,
                "risk_score": float(risk_score),
                "explanation": exp.as_list(label=1)
            }

        # Graphique LIME
        fig = exp.as_pyplot_figure(label=1)
        fig.set_size_inches(12, 6)
        plt.title(
            f'LIME — GRU Semaine {cp}\n'
            f'Étudiant {case_label} (risk={risk_score:.3f})',
            fontweight='bold'
        )
        plt.tight_layout()
        plt.show()

        # Top features LIME → regrouper par feature
        lime_list = exp.as_list(label=1)
        print(f"  Top features LIME (sem.{cp}) :")
        for feat, weight in lime_list[:6]:
            direction = '↑ risque' if weight > 0 else '↓ risque'
            print(f"    {feat:<50} → {weight:+.4f} ({direction})")

In [ ]:
with open(
    OUTPUT_PATH + "xai/lime_results.pkl",
    "wb"
) as f:
    pickle.dump(lime_results, f)

print("LIME sauvegardé")

In [ ]:
# ════════════════════════════════════════════════════════════
# PARTIE D — TRADUCTION PÉDAGOGIQUE GRU (VERSION CORRIGÉE)
# ════════════════════════════════════════════════════════════
print("\n" + "=" * 60)
print("PARTIE D — Traduction pédagogique GRU")
print("=" * 60)

OPTIMAL_THRESHOLD = 0.35

pedagogical_reports = []

def generate_pedagogical_explanation_gru(
    student_idx, shap_arr, X_seq_student,
    risk_score, checkpoint, model_name="GRU"
):
    """
    Génère un rapport pédagogique pour un étudiant
    basé sur les SHAP values du GRU.

    shap_arr      : (cp, F) SHAP values pour cet étudiant
    X_seq_student : (cp, F) valeurs des features (normalisées)

    """
    print(f"\n{'='*55}")
    print(f"RAPPORT PÉDAGOGIQUE — {model_name} (Semaine {checkpoint})")
    print(f"{'='*55}")

    risk_label = 'À risque' if risk_score >= OPTIMAL_THRESHOLD \
                 else 'Faible risque'
    print(f"Score de risque : {risk_score:.1%} ({risk_label})")

    # ── importance = |SHAP| moyen, mais signe = SHAP signé
    shap_per_feat_abs  = np.abs(shap_arr).mean(axis=0)   # (F,) — pour le ranking
    shap_per_feat_sign = shap_arr.mean(axis=0)            # (F,) — pour la direction

    top_idx = np.argsort(shap_per_feat_abs)[::-1][:6]

    # Valeur moyenne de la feature sur les semaines (normalisée)
    feat_means = X_seq_student.mean(axis=0)  # (F,)

    # ── seuil relatif basé sur le percentile 75 des |SHAP|
    threshold_high = np.percentile(shap_per_feat_abs, 75)

    print(f"\nFacteurs déterminants (semaines 1→{checkpoint}) :")

    messages = []
    for i in top_idx:
        feat_name  = readable_seq_list[i]
        shap_abs   = shap_per_feat_abs[i]
        shap_sign  = shap_per_feat_sign[i] 
        feat_val   = feat_means[i]

        # ── direction basée sur le SHAP signé
        direction = 'high' if shap_sign > 0 else 'low'
        rule      = PEDAGOGICAL_RULES_GRU.get(feat_name, {})
        msg_text  = rule.get(
            direction,
            f"{feat_name} ({'↑' if shap_sign > 0 else '↓'} risque)"
        )

        # ── icône cohérente avec la direction réelle
        # Rouge = augmente le risque, Vert = réduit le risque

        if abs(shap_sign) < 1e-3:
          direction_icon = '➡️ neutre'
          msg_text = f"{feat_name} — impact négligeable"

        elif shap_sign > 0:
            direction_icon = '📈 ↑ risque'
        else:
            direction_icon = '📉 ↓ risque'



        # ── niveau d'impact relatif
        impact = '🔴 Fort impact' if shap_abs >= threshold_high \
                 else '🟡 Impact modéré'

        # ── CORRECTION 6 : icône basée uniquement sur le SHAP
        if shap_sign > 1e-3:
          risk_icon = '🔺'   # augmente le risque

        elif shap_sign < -1e-3:
          risk_icon = '🔻'   # réduit le risque

        else:
          risk_icon = '➡️'   # neutre

        # Supprimer les anciens emojis pédagogiques
        msg_text_clean = (
            msg_text
            .replace('✅ ', '')
            .replace('⚠️ ', '')
        )

        print(
            f"  {impact} | {direction_icon} | "
            f"{risk_icon} {msg_text_clean}"
        )


        print(f"           Valeur moy. (norm.) : {feat_val:+.2f} | "
              f"SHAP moy. : {shap_sign:+.4f}")
        messages.append(msg_text_clean)

    # ── Semaine la plus critique
    shap_per_week = np.abs(shap_arr).mean(axis=1)   # (cp,)
    worst_week    = int(np.argmax(shap_per_week)) + 1
    print(f"\nSemaine la plus critique : Semaine {worst_week}")

    # ── Recommandation
    print(f"\nRecommandation :")
    if risk_score >= 0.7:
        print("  🔴 INTERVENTION URGENTE — Contacter l'étudiant "
              "immédiatement")
    elif risk_score >= OPTIMAL_THRESHOLD:
        print("  🟡 SURVEILLANCE RENFORCÉE — Proposer un suivi "
              "personnalisé et vérifier les soumissions")
    else:
        print("  🟢 FAIBLE RISQUE — Maintenir le suivi standard")

    report = {
        "risk_score": float(risk_score),
        "messages": messages
    }

    return report


# ════════════════════════════════════════════════════════════
# Génération des rapports pour chaque checkpoint
# ════════════════════════════════════════════════════════════
for cp in CHECKPOINTS:
    print(f"\n{'='*60}")
    print(f"RAPPORTS PÉDAGOGIQUES — GRU Semaine {cp}")
    print('='*60)

    model_cp    = checkpoint_results[cp]['model']
    model_cp.eval()
    risk_preds  = checkpoint_results[cp]['preds']
    true_labels = checkpoint_results[cp]['true']
    shap_arr_cp = shap_results[cp]['shap_arr']   # (N, cp, F)

    # Sélectionner 3 cas représentatifs
    cases = [
        (
            np.where(
                (risk_preds[:200] >= 0.7) &
                (true_labels[:200] == 1)
            )[0],
            'À Risque (score élevé)',
            'max'   # ← on veut le score le PLUS élevé
        ),
        (
            np.where(
                (risk_preds[:200] >= OPTIMAL_THRESHOLD) &
                (risk_preds[:200] < 0.7) &
                (true_labels[:200] == 1)
            )[0],
            'À Risque (score modéré)',
            'mid'   # ← on veut le score le plus proche du milieu de [seuil, 0.7]
        ),
        (
            np.where(
                (risk_preds[:200] < OPTIMAL_THRESHOLD) &
                (true_labels[:200] == 0)
            )[0],
            'Non À Risque',
            'min'   # ← on veut le score le PLUS faible
        ),
    ]

    for idx_arr, case_label, selection in cases:
        if len(idx_arr) == 0:
            print(f"\n  [!] Aucun étudiant trouvé pour le cas : {case_label}")
            continue

        # ── CORRECTION 6 : choisir l'exemple le plus représentatif
        # au lieu de toujours prendre idx_arr[0]
        scores_subset = risk_preds[:200][idx_arr]

        if selection == 'max':
            # Vrai positif le plus confiant
            idx = idx_arr[np.argmax(scores_subset)]
        elif selection == 'min':
            # Vrai négatif le plus confiant
            idx = idx_arr[np.argmin(scores_subset)]
        else:
            # Score modéré : le plus proche du centre de [seuil, 0.7]
            mid_target = (OPTIMAL_THRESHOLD + 0.7) / 2
            idx = idx_arr[np.argmin(np.abs(scores_subset - mid_target))]

        risk_score   = float(risk_preds[idx])
        shap_student = shap_arr_cp[idx]       # (cp, F)
        X_student    = X_test[idx, :cp, :]    # (cp, F)

        generate_pedagogical_explanation_gru(
            student_idx   = idx,
            shap_arr      = shap_student,
            X_seq_student = X_student,
            risk_score    = risk_score,
            checkpoint    = cp,
            model_name    = f"GRU Early Warning — Cas {case_label}"
        )

        pedagogical_reports.append({
            "model": "gru",
            "checkpoint": cp,
            "case": case_label,
            "report": report
        })

In [ ]:
with open(
    OUTPUT_PATH + "xai/pedagogical_reports.pkl",
    "wb"
) as f:
    pickle.dump(pedagogical_reports, f)

print("Rapports pédagogiques sauvegardés")

In [ ]:
print("\nSHAP GRU par checkpoint :")
for cp in CHECKPOINTS:
    top3_gru = np.argsort(
        shap_results[cp]['shap_global']
    )[::-1][:3]
    print(f"\n  Semaine {cp} :")
    for i in top3_gru:
        print(f"    {readable_seq_list[i]:<35} "
              f"SHAP={shap_results[cp]['shap_global'][i]:.4f}")

In [ ]:
# ════════════════════════════════════════════════════════════
# PARTIE E — COMPARAISON SHAP STATIQUE vs TEMPOREL
# ════════════════════════════════════════════════════════════
print("\n" + "=" * 60)
print("PARTIE E — Comparaison SHAP Statique vs Temporel")
print("=" * 60)

# Top features XGBoost (statique)
top_xgb = pd.DataFrame({
    'feature'   : readable_names,
    'shap_mean' : np.abs(shap_vals_xgb).mean(axis=0),
    'modele'    : 'XGBoost Jour 0'
}).nlargest(8, 'shap_mean')

# Top features GRU (temporel)
top_gru = pd.DataFrame({
    'feature'   : readable_seq,
    'shap_mean' : shap_gru_global,
    'modele'    : 'GRU Temporel'
}).nlargest(8, 'shap_mean')

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle('Comparaison SHAP — Modèle Statique vs Temporel',
             fontweight='bold', fontsize=13)

for ax, df, color, title in zip(
    axes,
    [top_xgb, top_gru],
    ['#16A34A', '#7C3AED'],
    ['XGBoost Jour 0\n(Features démographiques)',
     'GRU Temporel\n(Features VLE hebdomadaires)']
):
    sorted_df = df.sort_values('shap_mean')
    ax.barh(sorted_df['feature'], sorted_df['shap_mean'],
            color=color, alpha=0.85)
    ax.set_title(title, fontweight='bold')
    ax.set_xlabel('|SHAP| moyen')
    ax.grid(alpha=0.3, axis='x')

plt.tight_layout()
plt.show()

In [ ]:
# ════════════════════════════════════════════════════════════
# RESUME FINAL
# ════════════════════════════════════════════════════════════
print("\n" + "=" * 60)
print("RÉSUMÉ M4 — XAI")
print("=" * 60)

print(f"\nSHAP XGBoost (Jour 0) :")
print(f"  Top 3 features :")
for i in top3_idx:
    print(f"    {readable_names[i]:<35} "


          f"SHAP={mean_abs_shap[i]:.4f}")

print(f"\nSHAP GRU (Temporel) :")
top3_gru_idx = np.argsort(shap_gru_global)[::-1][:3]
for i in top3_gru_idx:
    print(f"    {readable_seq[i]:<35} "
          f"SHAP={shap_gru_global[i]:.4f}")



In [ ]:
# =====================================================
# SAUVEGARDE SHAP XGBOOST
# =====================================================

np.save(
    OUTPUT_PATH + "xai/shap_values_xgb.npy",
    shap_vals_xgb
)

np.save(
    OUTPUT_PATH + "xai/xgb_test_features.npy",
    X_te
)

np.save(
    OUTPUT_PATH + "xai/xgb_test_labels.npy",
    y_te
)

with open(
    OUTPUT_PATH + "xai/xgb_feature_names.pkl",
    "wb"
) as f:
    pickle.dump(readable_names, f)

print("SHAP XGBoost sauvegardé")

In [ ]:
# =====================================================
# SAUVEGARDE SHAP GRU GLOBAL
# =====================================================

np.save(
    OUTPUT_PATH + "xai/shap_gru_full.npy",
    shap_values_gru
)

np.save(
    OUTPUT_PATH + "xai/shap_gru_global.npy",
    shap_gru_global
)

print("SHAP GRU global sauvegardé")